# BharatStock AI — V94 Multi-Stock Model Research

This notebook benchmarks Logistic Regression, Random Forest, Extra Trees, HistGradientBoosting, LSTM and GRU across seven Indian market instruments. It uses chronological walk-forward folds and a separate final 20% untouched holdout for each symbol. It does not deploy models into the live API.

**Target:** next-session direction (`Close[t+1] > Close[t]`). Results are research metrics, not trading advice or a promise of future performance.

In [ ]:
!pip -q install yfinance scikit-learn pandas numpy matplotlib torch


In [ ]:
import sys, os, json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED=42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


## 1. Multi-stock experiment configuration

The notebook downloads adjusted daily OHLCV data independently for each instrument. Symbols with insufficient data are reported as skipped; they are not silently treated as zero performance.

In [ ]:
SYMBOLS = ['^NSEI', 'RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'ICICIBANK.NS', 'SBIN.NS']
PERIOD = '10y'
FINAL_HOLDOUT_FRACTION = 0.20
MIN_ROWS = 450
print('Symbols:', SYMBOLS)
print('Historical period:', PERIOD, '| final untouched holdout:', FINAL_HOLDOUT_FRACTION)


## 2. Causal feature engineering

All predictors use current or past OHLCV rows. The final row has no known next-session label and is excluded.


In [ ]:
BASE=['RSI','MACD_pct','MACD_Signal_pct','SMA20_gap','SMA50_gap','EMA20_gap','ATR_pct','Return_5']
EXTRA=['Return_1','Return_10','Return_20','Volatility_5','Range_pct','Close_Location','Gap_pct','Volume_Ratio_20']

def make_features(raw):
    d=raw.copy().sort_index(); c,h,l,o,v=(pd.to_numeric(d[k],errors='coerce') for k in ['Close','High','Low','Open','Volume'])
    d['SMA20']=c.rolling(20).mean(); d['SMA50']=c.rolling(50).mean(); d['EMA20']=c.ewm(span=20,adjust=False).mean()
    delta=c.diff(); gain=delta.clip(lower=0).rolling(14).mean(); loss=-delta.clip(upper=0).rolling(14).mean()
    d['RSI']=(100-100/(1+gain/loss.replace(0,np.nan))).fillna(50).clip(0,100)
    macd=c.ewm(span=12,adjust=False).mean()-c.ewm(span=26,adjust=False).mean()
    d['MACD_pct']=macd/c.replace(0,np.nan); d['MACD_Signal_pct']=macd.ewm(span=9,adjust=False).mean()/c.replace(0,np.nan)
    d['SMA20_gap']=c/d['SMA20']-1; d['SMA50_gap']=c/d['SMA50']-1; d['EMA20_gap']=c/d['EMA20']-1
    prev=c.shift(1); tr=pd.concat([h-l,(h-prev).abs(),(l-prev).abs()],axis=1).max(axis=1)
    d['ATR_pct']=tr.rolling(14).mean()/c.replace(0,np.nan); d['Return_5']=c.pct_change(5)
    d['Return_1']=c.pct_change(); d['Return_10']=c.pct_change(10); d['Return_20']=c.pct_change(20)
    d['Volatility_5']=c.pct_change().rolling(5).std(); d['Range_pct']=(h-l)/c.replace(0,np.nan)
    d['Close_Location']=((c-l)/(h-l).replace(0,np.nan)).clip(0,1); d['Gap_pct']=o/prev.replace(0,np.nan)-1
    d['Volume_Ratio_20']=v/v.rolling(20).median().replace(0,np.nan)
    d['Target']=(c.shift(-1)>c).astype(float); d.loc[d.index[-1],'Target']=np.nan
    return d.replace([np.inf,-np.inf],np.nan)

data=make_features(raw)
print('Feature count:',len(BASE+EXTRA),'Rows with complete features:',data[BASE+EXTRA+['Target']].dropna().shape[0])


## 3. Classical ML feature ablation + model comparison

Each feature set uses identical expanding-window test periods. The final training row before each test fold is purged because its label depends on the first test row. Model configurations are fixed in advance; this is a conservative comparison, not exhaustive hyperparameter search.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import brier_score_loss, log_loss

FEATURE_SETS={'Original 8':BASE,'Expanded 16':BASE+EXTRA,'Momentum & volatility':['RSI','Return_1','Return_5','Return_10','Return_20','Volatility_5','ATR_pct'],'Trend only':['MACD_pct','MACD_Signal_pct','SMA20_gap','SMA50_gap','EMA20_gap']}
FACTORIES={
 'LogisticRegression':lambda:make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,class_weight='balanced',C=.1,random_state=42)),
 'RandomForest':lambda:RandomForestClassifier(n_estimators=180,max_depth=6,min_samples_leaf=8,max_features='sqrt',class_weight='balanced_subsample',random_state=42,n_jobs=-1),
 'ExtraTrees':lambda:ExtraTreesClassifier(n_estimators=180,max_depth=6,min_samples_leaf=8,max_features='sqrt',class_weight='balanced',random_state=43,n_jobs=-1),
 'HistGradientBoosting':lambda:HistGradientBoostingClassifier(max_iter=120,learning_rate=.04,max_leaf_nodes=7,l2_regularization=2,random_state=44)}

def walk_forward(df, features, n_folds=5):
    z=df[features+['Target']].dropna(); n=len(z); initial=max(100,int(n*.5)); remain=n-initial
    if remain<n_folds*12: return []
    sizes=np.full(n_folds,remain//n_folds,dtype=int); sizes[:remain%n_folds]+=1; cursor=initial; rows=[]
    for fold,size in enumerate(sizes,1):
        tr=z.iloc[:cursor-1]; te=z.iloc[cursor:cursor+int(size)]; cursor+=int(size)
        if len(tr)<80 or te.empty: continue
        ytr=tr.Target.astype(int); yte=te.Target.astype(int); base=int(ytr.value_counts().idxmax())
        row={'fold':fold,'baseline_accuracy':accuracy_score(yte,np.full(len(te),base))}
        for name,make in FACTORIES.items():
            m=make(); m.fit(tr[features],ytr); pred=m.predict(te[features])
            row[name]={'accuracy':accuracy_score(yte,pred),'balanced_accuracy':balanced_accuracy_score(yte,pred),'f1':f1_score(yte,pred,zero_division=0)}
        rows.append(row)
    return rows

results={}
for fs,features in FEATURE_SETS.items():
    folds=walk_forward(data,features)
    results[fs]=folds
    print('\n',fs,'folds=',len(folds))
    for model in FACTORIES:
        vals=[r[model]['accuracy'] for r in folds]
        print(model, 'mean accuracy=',round(float(np.mean(vals)),4) if vals else None)
    b=[r['baseline_accuracy'] for r in folds]
    print('Majority baseline=',round(float(np.mean(b)),4) if b else None)


In [ ]:
# Plot mean walk-forward accuracy by feature set and model
summary=[]
for fs,folds in results.items():
    for model in list(FACTORIES)+['Majority baseline']:
        vals=[r[model]['accuracy'] if model!='Majority baseline' else r['baseline_accuracy'] for r in folds]
        summary.append({'Feature set':fs,'Model':model,'Mean accuracy':np.mean(vals) if vals else np.nan})
plot_df=pd.DataFrame(summary)
if not plot_df.empty:
    pivot=plot_df.pivot(index='Feature set',columns='Model',values='Mean accuracy')
    pivot.plot(kind='bar',figsize=(12,5),ylim=(0,1),title='Chronological walk-forward accuracy')
    plt.ylabel('Accuracy'); plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.show()


## 4. LSTM and GRU sequence models (optional)

The sequence models are trained separately inside each chronological fold. The scaler is fitted only on the training segment. Early stopping is not used on the outer test fold. The last part of the training window is used as an internal chronological validation segment for epoch selection.


In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix

class SeqNet(nn.Module):
    def __init__(self, kind, n_features, hidden=32):
        super().__init__(); self.kind=kind
        self.rnn=(nn.LSTM(n_features,hidden,batch_first=True) if kind=='LSTM' else nn.GRU(n_features,hidden,batch_first=True))
        self.head=nn.Sequential(nn.Dropout(.2),nn.Linear(hidden,1))
    def forward(self,x):
        out,_=self.rnn(x); return self.head(out[:,-1,:]).squeeze(-1)

def seq_arrays(x,y,lookback):
    xs=[]; ys=[]
    for end in range(lookback-1,len(x)):
        xs.append(x[end-lookback+1:end+1]); ys.append(y[end])
    return np.asarray(xs,dtype=np.float32),np.asarray(ys,dtype=np.float32)

def fit_sequence_fold(train_df,test_df,features,kind,lookback=20,epochs=25):
    scaler=StandardScaler().fit(train_df[features]) # training rows only
    xtr=scaler.transform(train_df[features]); ytr=train_df.Target.to_numpy(dtype=np.float32)
    xte=scaler.transform(test_df[features]); yte=test_df.Target.to_numpy(dtype=np.int64)
    X,Y=seq_arrays(xtr,ytr,lookback); Xtest,Ytest=seq_arrays(np.vstack([xtr[-(lookback-1):],xte]),np.r_[ytr[-(lookback-1):],yte],lookback)
    if len(X)<80 or len(Xtest)==0: return None
    split=max(1,int(len(X)*.85)); Xfit,Yfit=X[:split],Y[:split]; Xval,Yval=X[split:],Y[split:]
    if len(Xval)==0: Xval,Yval=Xfit[-1:],Yfit[-1:]
    model=SeqNet(kind,X.shape[-1]).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=.001,weight_decay=1e-4); lossfn=nn.BCEWithLogitsLoss()
    loader=DataLoader(TensorDataset(torch.tensor(Xfit),torch.tensor(Yfit)),batch_size=32,shuffle=False)
    best=float('inf'); best_state=None; patience=5; stale=0
    for epoch in range(epochs):
        model.train()
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); loss=lossfn(model(xb),yb); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        model.eval()
        with torch.no_grad(): val=float(lossfn(model(torch.tensor(Xval).to(DEVICE)),torch.tensor(Yval).to(DEVICE)).item())
        if val<best: best=val; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
        else: stale+=1
        if stale>=patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad(): probs=torch.sigmoid(model(torch.tensor(Xtest).to(DEVICE))).cpu().numpy()
    pred=(probs>=.5).astype(int)
    majority=int(pd.Series(ytr).value_counts().idxmax())
    baseline_pred=np.full(len(Ytest), majority, dtype=int)
    try:
        auc=float(roc_auc_score(Ytest, probs)) if len(np.unique(Ytest)) == 2 else None
    except Exception:
        auc=None
    return {
        'accuracy':float(accuracy_score(Ytest,pred)),
        'balanced_accuracy':float(balanced_accuracy_score(Ytest,pred)),
        'precision':float(precision_score(Ytest,pred,zero_division=0)),
        'recall':float(recall_score(Ytest,pred,zero_division=0)),
        'f1':float(f1_score(Ytest,pred,zero_division=0)),
        'roc_auc':auc,
        'baseline_accuracy':float(accuracy_score(Ytest,baseline_pred)),
        'test_rows':int(len(Ytest)),
        'epochs_run':int(epoch+1),
        'confusion_matrix':confusion_matrix(Ytest,pred).tolist()
    }

# Use the expanded feature set; each test fold is still strictly chronological.
seq_results={'LSTM':[],'GRU':[]}
features=BASE+EXTRA; z=data[features+['Target']].dropna(); n=len(z); initial=max(100,int(n*.5)); remaining=n-initial; nf=5
if remaining>=nf*12:
    sizes=np.full(nf,remaining//nf,dtype=int); sizes[:remaining%nf]+=1; cursor=initial
    for fold,size in enumerate(sizes,1):
        tr=z.iloc[:cursor-1]; te=z.iloc[cursor:cursor+int(size)]; cursor+=int(size)
        if len(tr)<120 or te.empty: continue
        for kind in ['LSTM','GRU']:
            try:
                result=fit_sequence_fold(tr,te,features,kind)
                if result: result['fold']=fold; seq_results[kind].append(result)
            except Exception as e: print(kind,'fold',fold,'skipped:',repr(e))
for kind,folds in seq_results.items():
    if not folds:
        print(kind, ': not enough data to evaluate')
        continue
    print(
        kind,
        '| folds:', len(folds),
        '| accuracy:', round(float(np.mean([x['accuracy'] for x in folds])), 4),
        '| balanced accuracy:', round(float(np.mean([x['balanced_accuracy'] for x in folds])), 4),
        '| F1:', round(float(np.mean([x['f1'] for x in folds])), 4),
        '| majority baseline:', round(float(np.mean([x['baseline_accuracy'] for x in folds])), 4)
    )


## 5. Seven-symbol benchmark and untouched future-period evaluation

For each symbol, the last 20% of complete labelled observations is reserved as a final holdout and is not used for model selection or fold training. Classical models are evaluated on the earlier development segment with expanding chronological folds, then fitted once on the development segment and evaluated on the untouched holdout. LSTM/GRU are evaluated on the same development folds and final holdout. Scaling is fitted on training rows only.

The final holdout is a single historical period, not a guarantee of future performance. Do not repeatedly tune against it.

In [ ]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score, brier_score_loss, log_loss
import traceback

def metric_pack(y_true, pred, prob, baseline_pred):
    out = {
        'accuracy': float(accuracy_score(y_true, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, pred)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'baseline_accuracy': float(accuracy_score(y_true, baseline_pred)),
        'confusion_matrix': confusion_matrix(y_true, pred).tolist()
    }
    if prob is not None:
        out['roc_auc'] = float(roc_auc_score(y_true, prob)) if len(np.unique(y_true)) == 2 else None
        out['brier_score'] = float(brier_score_loss(y_true, prob))
        out['log_loss'] = float(log_loss(y_true, np.clip(prob, 1e-6, 1-1e-6), labels=[0,1]))
    else:
        out.update({'roc_auc': None, 'brier_score': None, 'log_loss': None})
    return out

def final_holdout_classical(z, feature_set, features, holdout_fraction=FINAL_HOLDOUT_FRACTION):
    cut = int(len(z) * (1-holdout_fraction))
    dev, hold = z.iloc[:cut], z.iloc[cut:]
    # Purge the final development row because its next-session target overlaps the first holdout date.
    train = dev.iloc[:-1]
    if len(train) < 150 or len(hold) < 30:
        return {}
    ytr = train.Target.astype(int).to_numpy()
    yte = hold.Target.astype(int).to_numpy()
    base = int(pd.Series(ytr).value_counts().idxmax())
    rows = {}
    for name, factory in FACTORIES.items():
        model = factory()
        model.fit(train[features], ytr)
        pred = model.predict(hold[features]).astype(int)
        prob = model.predict_proba(hold[features])[:,1] if hasattr(model, 'predict_proba') else None
        rows[name] = metric_pack(yte, pred, prob, np.full(len(yte), base))
    return {'development_rows': int(len(train)), 'holdout_rows': int(len(hold)), 'models': rows}

def run_symbol(symbol):
    raw_symbol = yf.Ticker(symbol).history(period=PERIOD, auto_adjust=True)
    raw_symbol = raw_symbol.rename(columns={c:str(c).title() for c in raw_symbol.columns})
    raw_symbol = raw_symbol[['Open','High','Low','Close','Volume']].dropna()
    if len(raw_symbol) < MIN_ROWS:
        raise ValueError(f'Insufficient OHLCV rows: {len(raw_symbol)} < {MIN_ROWS}')
    d = make_features(raw_symbol)
    z = d[BASE+EXTRA+['Target']].dropna()
    if len(z) < MIN_ROWS-60:
        raise ValueError(f'Insufficient complete labelled rows: {len(z)}')
    # Walk-forward development-only folds: final 20% is excluded before fold generation.
    dev_cut = int(len(z)*(1-FINAL_HOLDOUT_FRACTION))
    development = z.iloc[:dev_cut]
    classical_cv = {}
    for fs, feats in FEATURE_SETS.items():
        classical_cv[fs] = walk_forward(development, feats, n_folds=5)
    # Sequence walk-forward development folds.
    sequence_cv = {'LSTM': [], 'GRU': []}
    seq_features = BASE+EXTRA
    seq_z = development[seq_features+['Target']].dropna()
    n = len(seq_z); initial = max(100, int(n*.5)); remaining = n-initial; nf=5
    if remaining >= nf*12:
        sizes = np.full(nf, remaining//nf, dtype=int); sizes[:remaining%nf] += 1
        cursor = initial
        for fold, size in enumerate(sizes, 1):
            tr = seq_z.iloc[:cursor-1]
            te = seq_z.iloc[cursor:cursor+int(size)]
            cursor += int(size)
            if len(tr) < 120 or te.empty:
                continue
            for kind in ['LSTM','GRU']:
                result = fit_sequence_fold(tr, te, seq_features, kind)
                if result:
                    result['fold'] = fold
                    sequence_cv[kind].append(result)
    # Untouched final holdout: only one final evaluation after the CV research.
    holdout = final_holdout_classical(z, 'Expanded 16', seq_features)
    cut = int(len(z)*(1-FINAL_HOLDOUT_FRACTION))
    dev_z, hold_z = z.iloc[:cut], z.iloc[cut:]
    # Purge one row before final holdout; sequence helper uses preceding training rows only as lookback context.
    seq_holdout = {}
    train_seq = dev_z.iloc[:-1]
    if len(train_seq) >= 150 and len(hold_z) >= 30:
        for kind in ['LSTM','GRU']:
            r = fit_sequence_fold(train_seq, hold_z, seq_features, kind)
            seq_holdout[kind] = r
    return {
        'symbol': symbol,
        'raw_rows': int(len(raw_symbol)),
        'complete_rows': int(len(z)),
        'date_start': str(z.index.min().date()),
        'date_end': str(z.index.max().date()),
        'development_walk_forward': classical_cv,
        'sequence_walk_forward': sequence_cv,
        'untouched_final_holdout_classical': holdout,
        'untouched_final_holdout_sequence': seq_holdout
    }

multi_stock_results = {}
skipped_symbols = {}
for symbol in SYMBOLS:
    print('\n' + '='*70 + '\nRunning:', symbol)
    try:
        multi_stock_results[symbol] = run_symbol(symbol)
        print('Completed:', symbol)
    except Exception as exc:
        skipped_symbols[symbol] = repr(exc)
        print('SKIPPED:', symbol, '|', repr(exc))

print('\nFinished symbols:', list(multi_stock_results))
print('Skipped symbols:', skipped_symbols)

## 6. Consolidated reports

The exports preserve per-symbol fold details, final holdout results, skipped symbols and caveats. The CSV is a flattened summary for easier review.

In [ ]:
summary_rows = []
for symbol, report in multi_stock_results.items():
    for feature_set, folds in report['development_walk_forward'].items():
        for model_name in FACTORIES:
            vals = [f[model_name] for f in folds]
            if vals:
                summary_rows.append({
                    'symbol': symbol, 'evaluation': 'development_walk_forward',
                    'feature_set': feature_set, 'model': model_name, 'folds': len(vals),
                    'mean_accuracy': float(np.mean([v['accuracy'] for v in vals])),
                    'std_accuracy': float(np.std([v['accuracy'] for v in vals])),
                    'mean_balanced_accuracy': float(np.mean([v['balanced_accuracy'] for v in vals])),
                    'mean_f1': float(np.mean([v['f1'] for v in vals])),
                    'mean_baseline_accuracy': float(np.mean([f['baseline_accuracy'] for f in folds]))
                })
    for model_name, folds in report['sequence_walk_forward'].items():
        if folds:
            summary_rows.append({
                'symbol': symbol, 'evaluation': 'development_walk_forward',
                'feature_set': 'Expanded 16 (sequence)', 'model': model_name, 'folds': len(folds),
                'mean_accuracy': float(np.mean([v['accuracy'] for v in folds])),
                'std_accuracy': float(np.std([v['accuracy'] for v in folds])),
                'mean_balanced_accuracy': float(np.mean([v['balanced_accuracy'] for v in folds])),
                'mean_f1': float(np.mean([v['f1'] for v in folds])),
                'mean_baseline_accuracy': float(np.mean([v['baseline_accuracy'] for v in folds]))
            })
    hold = report.get('untouched_final_holdout_classical') or {}
    for model_name, metrics in hold.get('models', {}).items():
        summary_rows.append({
            'symbol': symbol, 'evaluation': 'untouched_final_holdout',
            'feature_set': 'Expanded 16', 'model': model_name, 'folds': 1,
            'mean_accuracy': metrics['accuracy'], 'std_accuracy': 0.0,
            'mean_balanced_accuracy': metrics['balanced_accuracy'],
            'mean_f1': metrics['f1'], 'mean_baseline_accuracy': metrics['baseline_accuracy']
        })
    for model_name, metrics in report.get('untouched_final_holdout_sequence', {}).items():
        if metrics:
            summary_rows.append({
                'symbol': symbol, 'evaluation': 'untouched_final_holdout',
                'feature_set': 'Expanded 16 (sequence)', 'model': model_name, 'folds': 1,
                'mean_accuracy': metrics['accuracy'], 'std_accuracy': 0.0,
                'mean_balanced_accuracy': metrics['balanced_accuracy'],
                'mean_f1': metrics['f1'], 'mean_baseline_accuracy': metrics['baseline_accuracy']
            })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.sort_values(['evaluation','symbol','mean_balanced_accuracy'], ascending=[True,True,False]) if not summary_df.empty else summary_df)

final_report = {
    'project': 'BharatStock AI V94',
    'target': 'next-session close direction',
    'symbols_requested': SYMBOLS,
    'symbols_completed': list(multi_stock_results),
    'symbols_skipped': skipped_symbols,
    'holdout_fraction': FINAL_HOLDOUT_FRACTION,
    'results_by_symbol': multi_stock_results,
    'summary_rows': summary_rows,
    'caveats': [
        'Historical results are not future guarantees.',
        'The final holdout is a single historical period and must not be repeatedly used for tuning.',
        'No transaction costs, slippage, taxes or order execution simulation are included.',
        'Classifier scores are not necessarily calibrated trade-success probabilities.',
        'Only symbols with sufficient data are included; inspect skipped_symbols.',
        'No model is automatically deployed to the live API.'
    ]
}
def json_safe(x):
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.ndarray,)): return x.tolist()
    if isinstance(x, (pd.Timestamp,)): return x.isoformat()
    raise TypeError(f'Not JSON serializable: {type(x).__name__}')

with open('bharatstock_ai_v94_multi_stock_results.json','w') as f:
    json.dump(final_report, f, indent=2, default=json_safe)
summary_df.to_csv('bharatstock_ai_v94_model_comparison.csv', index=False)
print('Saved bharatstock_ai_v94_multi_stock_results.json')
print('Saved bharatstock_ai_v94_model_comparison.csv')